# SkinFLNet++ — ISIC 2018 experiments (local Jupyter or Google Colab)

Run sections **in order** the first time on Colab: **setup → install → download ISIC 2018 → manifests → imports →** individual experiments.

**Runtime**

- **Local:** open from `notebooks/`; setup finds repo root via `pyproject.toml`.
- **Colab + Drive:** code on Drive; **`DATA_ROOT`** on session disk (`/content/skinfl_data`).

**Before you start**

- **GPU** recommended on Colab.
- **ISIC 2018 Task 3:** 7 classes; official train / val / test in `manifest.csv`.
- **Outputs:** metrics under `results/isic2018/<run_name>/`, figures under `figures/isic2018/` (separate from 2019).
- **`SKINFL_DROP_MISSING_IMAGES`:** auto-on for `colab_drive`; use `DROP_MISSING_IMAGES = False` after full download.



In [1]:
from __future__ import annotations

import logging
import os
import sys
from pathlib import Path

if sys.version_info < (3, 11):
    raise RuntimeError(
        f"This project requires Python >= 3.11 (pyproject.toml). Got: {sys.version}"
    )


def _in_colab() -> bool:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except ImportError:
        return False


def _runtime_mode() -> str:
    """local = this machine; colab_drive = mount Drive and use COLAB_DRIVE_ROOT."""
    env = os.environ.get("SKINFL_RUNTIME", "").strip().lower()
    if env in ("local", "colab_drive"):
        return env
    return "colab_drive" if _in_colab() else "local"


# Editable: where the repo lives on Google Drive (must contain pyproject.toml, scripts/, …)
COLAB_DRIVE_ROOT = Path("/content/drive/MyDrive/SkinFL")

# VM-local datasets on Colab (fast I/O); never under Drive — see setup + download cells.
COLAB_SESSION_DATA_ROOT = Path("/content/skinfl_data")

# Local: set to a Path to pin the repo root, or None to search from cwd / notebooks/
PROJECT_ROOT_OVERRIDE: Path | None = None

# Drive/Colab: True skips manifest rows whose image file is missing (typical for partial uploads).
# False = strict: require every training image path. Recommended after a full official ISIC session download.
# None = auto: True on colab_drive only (backward-compatible with partial uploads).
DROP_MISSING_IMAGES: bool | None = None


def _discover_project_root() -> Path:
    start = Path.cwd().resolve()
    candidates = [start]
    if start.name == "notebooks":
        candidates.append(start.parent)
    for cand in candidates:
        if (cand / "pyproject.toml").is_file() and (cand / "src").is_dir():
            return cand
    for parent in start.parents:
        if (parent / "pyproject.toml").is_file() and (parent / "src").is_dir():
            return parent
    raise FileNotFoundError(
        "Could not find SkinFL repo (pyproject.toml + src/). "
        "cd to repo root or notebooks/, or set PROJECT_ROOT_OVERRIDE, "
        "or SKINFL_RUNTIME=colab_drive on Colab."
    )


RUNTIME = _runtime_mode()

if RUNTIME == "colab_drive":
    try:
        from google.colab import drive
    except ImportError as e:
        raise SystemExit(
            'SKINFL_RUNTIME=colab_drive but not in Colab. '
            "Unset SKINFL_RUNTIME or use local Jupyter."
        ) from e
    drive.mount("/content/drive")
    PROJECT_ROOT = COLAB_DRIVE_ROOT
else:
    PROJECT_ROOT = PROJECT_ROOT_OVERRIDE or _discover_project_root()

if RUNTIME == "colab_drive":
    DATA_ROOT = COLAB_SESSION_DATA_ROOT
else:
    DATA_ROOT = PROJECT_ROOT / "data"

# ISIC 2018 runs are isolated from 2019 under results/isic2018/ and figures/isic2018/
RESULTS_DIR = PROJECT_ROOT / "results" / "isic2018"
FIGURES_DIR = PROJECT_ROOT / "figures" / "isic2018"

DATA_ROOT.mkdir(parents=True, exist_ok=True)
for path in (RESULTS_DIR, FIGURES_DIR):
    path.mkdir(parents=True, exist_ok=True)

if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError(
        f"Invalid PROJECT_ROOT: {PROJECT_ROOT} (pyproject.toml missing)."
    )

os.chdir(PROJECT_ROOT)

_repo_root = str(PROJECT_ROOT.resolve())
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

_drop = (RUNTIME == "colab_drive") if DROP_MISSING_IMAGES is None else DROP_MISSING_IMAGES
if _drop:
    os.environ["SKINFL_DROP_MISSING_IMAGES"] = "1"
else:
    os.environ.pop("SKINFL_DROP_MISSING_IMAGES", None)

print("RUNTIME:", RUNTIME)
print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("DATA_ROOT:", DATA_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("FIGURES_DIR:", FIGURES_DIR)
print(
    "SKINFL_DROP_MISSING_IMAGES:",
    os.environ.get("SKINFL_DROP_MISSING_IMAGES", "(unset — strict image checks)"),
)
if _drop:
    print(
        "Note: rows without a file on disk are dropped (subset run). "
        "Set DROP_MISSING_IMAGES=False after a full extract (official ISIC download cell) for strict counts."
    )


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RUNTIME: colab_drive
PROJECT_ROOT: /content/drive/MyDrive/SkinFL
DATA_ROOT: /content/skinfl_data
RESULTS_DIR: /content/drive/MyDrive/SkinFL/results/isic2018
FIGURES_DIR: /content/drive/MyDrive/SkinFL/figures/isic2018
SKINFL_DROP_MISSING_IMAGES: 1
Note: rows without a file on disk are dropped (subset run). Set DROP_MISSING_IMAGES=False after a full extract (official ISIC download cell) for strict counts.


In [2]:
%pip install -q -e ".[dev]"


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for skinflnet-plus (pyproject.toml) ... done


In [3]:
# ISIC 2018 Task 3: official challenge files -> DATA_ROOT/ISIC2018/

import shutil
import zipfile
from urllib.request import Request, urlopen

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None


def _download_file(url: str, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    req = Request(url, headers={"User-Agent": "SkinFL-ISIC-fetch/1.0"})
    with urlopen(req) as resp:  # noqa: S310
        hdr = resp.headers.get("Content-Length")
        total = int(hdr) if hdr is not None and hdr.isdigit() else None
        bs = 256 * 1024
        if tqdm is None:
            with open(dest, "wb") as f:
                while True:
                    chunk = resp.read(bs)
                    if not chunk:
                        break
                    f.write(chunk)
            return
        with open(dest, "wb") as f, tqdm(
            desc=dest.name[:48], total=total, unit="iB", unit_scale=True, unit_divisor=1024, miniters=1,
        ) as bar:
            while True:
                chunk = resp.read(bs)
                if not chunk:
                    break
                f.write(chunk)
                bar.update(len(chunk))


def _extract_csv_from_zip(zip_path: Path, dest_csv: Path) -> None:
    with zipfile.ZipFile(zip_path, "r") as zf:
        csv_members = [m for m in zf.namelist() if m.lower().endswith(".csv")]
        if not csv_members:
            raise FileNotFoundError(f"No CSV in {zip_path}")
        with zf.open(csv_members[0]) as src, open(dest_csv, "wb") as out:
            shutil.copyfileobj(src, out)


urls = [
    "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task3_Training_Input.zip",
    "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task3_Training_GroundTruth.zip",
    "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task3_Training_LesionGroupings.csv",
    "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task3_Validation_Input.zip",
    "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task3_Validation_GroundTruth.zip",
    "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task3_Test_Input.zip",
    "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task3_Test_GroundTruth.zip",
]

FORCE_ISIC_REDOWNLOAD = False


def _copy_repo_fallback_csvs(isic_dest: Path) -> None:
    root = PROJECT_ROOT
    pairs = [
        (root / "ISIC2018_Task3_Training_GroundTruth.csv", isic_dest / "ISIC2018_Task3_Training_GroundTruth.csv"),
        (root / "ISIC2018_Task3_Training_LesionGroupings (1).csv", isic_dest / "ISIC2018_Task3_Training_LesionGroupings.csv"),
        (root / "ISIC2018_Task3_Training_LesionGroupings.csv", isic_dest / "ISIC2018_Task3_Training_LesionGroupings.csv"),
    ]
    for src_p, dst_p in pairs:
        if src_p.is_file() and not dst_p.is_file():
            shutil.copy2(src_p, dst_p)
            print("  -> copied fallback", src_p.name)


def _isic2018_ready() -> bool:
    root = DATA_ROOT / "ISIC2018"
    train_in = root / "ISIC2018_Task3_Training_Input"
    csvs = [
        root / "ISIC2018_Task3_Training_GroundTruth.csv",
        root / "ISIC2018_Task3_Validation_GroundTruth.csv",
        root / "ISIC2018_Task3_Test_GroundTruth.csv",
        root / "ISIC2018_Task3_Training_LesionGroupings.csv",
    ]
    if not train_in.is_dir() or not all(p.is_file() for p in csvs):
        return False
    return any(train_in.glob("*.jpg"))


if RUNTIME != "colab_drive":
    print("SKIP: ISIC download is for Colab + Drive runtime only.")
    isic_dest = DATA_ROOT / "ISIC2018"
    isic_dest.mkdir(parents=True, exist_ok=True)
    _copy_repo_fallback_csvs(isic_dest)
elif _isic2018_ready() and not FORCE_ISIC_REDOWNLOAD:
    print("SKIP: ISIC2018 already present under", DATA_ROOT / "ISIC2018")
else:
    isic_dest = DATA_ROOT / "ISIC2018"
    staging = DATA_ROOT / ".cache_isic2018"
    isic_dest.mkdir(parents=True, exist_ok=True)
    staging.mkdir(parents=True, exist_ok=True)
    for url in urls:
        name = Path(url.rstrip("/")).name
        out = staging / name
        _download_file(url, out)
        if name.endswith(".csv"):
            shutil.copy2(out, isic_dest / name)
            print("  -> copied CSV to", isic_dest / name)
        elif name.endswith(".zip"):
            print("  -> extracting:", name, flush=True)
            if "GroundTruth" in name:
                csv_name = name.replace(".zip", ".csv")
                _extract_csv_from_zip(out, isic_dest / csv_name)
                print("  -> extracted CSV to", isic_dest / csv_name)
            else:
                with zipfile.ZipFile(out, "r") as zf:
                    infos = zf.infolist()
                    if tqdm is not None:
                        for m in tqdm(infos, desc=f"Unpack {name[:42]}", unit="file"):
                            zf.extract(m, isic_dest)
                    else:
                        zf.extractall(isic_dest)
    train_in = isic_dest / "ISIC2018_Task3_Training_Input"
    if not train_in.is_dir():
        raise FileNotFoundError(f"Missing after extract: {train_in}")
    _copy_repo_fallback_csvs(isic_dest)
    for p in staging.iterdir():
        p.unlink(missing_ok=True)
    print("Done. Training layout:", train_in.resolve())


SKIP: ISIC2018 already present under /content/skinfl_data/ISIC2018


In [4]:
from src.data.manifest_build import ensure_manifest_for_dataset

_root = Path(DATA_ROOT)
ensure_manifest_for_dataset(_root, "isic2018")


In [5]:
from src.fl.experiment_runner import run_experiment_from_yaml


def run_experiment(config_rel: str, **kwargs) -> None:
    """Run one YAML config in-process; writes under RESULTS_DIR and FIGURES_DIR.

    Extra kwargs match ``run_experiment_from_yaml`` (e.g. ``wandb_cli=True``,
    ``no_round_progress=True``, ``show_round_progress=False``).
    """
    path = Path(config_rel)
    if not path.is_absolute():
        path = PROJECT_ROOT / config_rel
    run_experiment_from_yaml(
        path,
        data_root=DATA_ROOT,
        results_dir=RESULTS_DIR,
        figures_dir=FIGURES_DIR,
        **kwargs,
    )


## Primary experiments

**Main ISIC2018 FL** (`isic2018_dirichlet.yaml`) and centralized upper bound.


In [ ]:
run_experiment("configs/isic2018_dirichlet.yaml")  # Main FL benchmark — Dirichlet α=100


In [6]:
run_experiment("configs/isic2018_centralized_upperbound.yaml")  # Pooled ISIC2018, no FL


INFO | Experiment: isic2018_centralized_upperbound
INFO | ISIC2018 manifest already exists at /content/skinfl_data/ISIC2018/manifest.csv (skipping)
INFO | IID partition: 10 clients, 10015 train images
INFO | Partition 'iid' applied (alpha=0.50, K=10, seed=42)
INFO | Partition figure saved: /content/drive/MyDrive/SkinFL/figures/isic2018/partition_isic2018_iid_K10_a0.5.png
INFO | Running CENTRALIZED upper-bound training
INFO | Centralized | 10015 train | 193 val | 1512 test samples | best_checkpoint_on=test
WARNING | centralized_best_on=test: early stopping and best_model.pth use the TEST set — biased vs real deployment; use for FL-protocol parity only.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You wi

model.safetensors:   0%|          | 0.00/554M [00:00<?, ?B/s]

INFO | Model: vgg16_bn | Classes: 7 | Params: 14,789,703 total (14,789,703 trainable) | Device: cuda
Centralized | isic2018_centralized_upperbound:   0%|          | 0/78 [00:00<?, ?epoch/s] 
Epoch 23/100: 100%|██████████| 313/313 [01:20<00:00,  4.20it/s]
Centralized | isic2018_centralized_upperbound:   1%|▏         | 1/78 [01:20<1:43:37, 80.75s/epoch] , tLoss=0.1025
Epoch 24/100: 100%|██████████| 313/313 [01:18<00:00,  4.01it/s]
                                                               INFO | Epoch 24/100 (round 12) | TrainLoss=0.0803 | TestAcc=0.7923 | TestF1=0.7108
Centralized | isic2018_centralized_upperbound:   1%|▏         | 1/78 [02:47<1:43:37, 80.75s/epoch] , vAcc=0.792, vF1=0.711INFO | Early stopping at epoch 24 (round 12)
Centralized | isic2018_centralized_upperbound:   1%|▏         | 1/78 [02:48<3:35:37, 168.02s/epoch] , vAcc=0.792, vF1=0.711
INFO | Centralized FINAL | Acc=0.8095 | F1=0.7485 | AUROC=0.9588 | ECE=0.0365
INFO | Done. Results in /content/drive/MyDrive/SkinF

## Partition ablations (ISIC2018, 10 clients)

Dirichlet α: 0.1, 0.5, 1.0. Headline: `isic2018_dirichlet.yaml` (α=100).


In [ ]:
run_experiment("configs/isic2018_ablation_partition_dirichlet_01.yaml")  # α=0.1


In [ ]:
run_experiment("configs/isic2018_ablation_partition_dirichlet_05.yaml")  # α=0.5


In [ ]:
run_experiment("configs/isic2018_ablation_partition_dirichlet_10.yaml")  # α=1.0


In [ ]:
# Optional: isic2018 partition_iid / patient YAMLs


## Client-count ablations


In [ ]:
run_experiment("configs/isic2018_ablation_nclients_5.yaml")  # 5 clients


In [ ]:
run_experiment("configs/isic2018_ablation_nclients_10.yaml")  # 10 clients


In [ ]:
run_experiment("configs/isic2018_ablation_nclients_20.yaml")  # 20 clients

## Strategy ablations


In [6]:
run_experiment("configs/isic2018_ablation_strategy_fedprox.yaml")


INFO | Experiment: isic2018_ablation_strategy_fedprox
INFO | ISIC2018 manifest already exists at /content/skinfl_data/ISIC2018/manifest.csv (skipping)
INFO | Dirichlet(alpha=0.50) partition: 10 clients, sizes={0: 2098, 1: 1356, 2: 2900, 3: 365, 4: 748, 5: 467, 6: 757, 7: 368, 8: 450, 9: 506}
INFO | Partition 'dirichlet' applied (alpha=0.50, K=10, seed=42)
INFO | Partition figure saved: /content/drive/MyDrive/SkinFL/figures/isic2018/partition_isic2018_dirichlet_K10_a0.5.png
INFO | Running FEDERATED simulation
INFO | Starting custom simulation | Run: isic2018_ablation_strategy_fedprox | Dataset: isic2018 | Backbone: vgg16_bn | Clients: 10 | Rounds: 50 | Strategy: fedprox | Partition: dirichlet(α=0.50)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in y

FL rounds | isic2018_ablation_strategy_fedprox:   0%|          | 0/50 [00:00<?, ?round/s] 

INFO | Round 1/50 | sampled clients (5/10): 1, 6, 5, 7, 8
INFO |   Client 1 training ...
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
WARNING | AUROC/AUPRC computation failed: Number of classes in y_true not equal to the number of columns in 'y_score'
INFO |   Client 1 done | train_loss=1.0140 | n_train=1084 | local_test_macro_f1=0.3399
INFO |   Client 6 training ...
INFO |   Client 6 done | train_loss=1.3384 | n_train=605 | local_test_macro_f1=0.3013
INFO |   Client 5 training ...
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
WARNING | AUROC/AUPRC computation failed: Number of classes in y_true not equal to the number of columns in 'y_score'
INFO |   Client 5 done | train_loss=1.3088 | n_train=373 | loc

In [ ]:
run_experiment("configs/isic2018_ablation_strategy_fedadam.yaml")


## Local-epoch ablations


In [ ]:
run_experiment("configs/isic2018_ablation_localepochs_1.yaml")


In [ ]:
run_experiment("configs/isic2018_ablation_localepochs_5.yaml")


In [ ]:
run_experiment("configs/isic2018_ablation_localepochs_10.yaml")


## Optional: viz / debug config

Skip unless you use `configs/test_viz.yaml` intentionally.


In [ ]:
# run_experiment("configs/test_viz.yaml")


## Optional: reporting tables and figures

Writes aggregate markdown tables and extra curves/matrices into the same `--results-dir` / `--figures-dir` as experiments (`docs/reproducibility.md` helpers).


In [ ]:
from src.report.tables import write_report_tables

report_md = PROJECT_ROOT / "REPORT_TABLES_ISIC2018.md"
write_report_tables(RESULTS_DIR, report_md)
print("Wrote:", report_md)


In [ ]:
from src.report.figures import generate_report_figures

generate_report_figures(RESULTS_DIR, FIGURES_DIR, "isic2018")

